In [2]:
from neuralhydrology.datautils.burn_areas import compute_burn_area
from pathlib import Path
from neuralhydrology.utils.eeutils import initialize_earth
import ee
from neuralhydrology.utils.config import Config
from neuralhydrology.modelzoo.revnetmodified import ModifiedRevnet

from neuralhydrology.datasetzoo.camelsusburn import CamelsUSBurn
from neuralhydrology.training.train import start_training
import pandas as pd
import numpy as np




In [3]:
initialize_earth(Path("../../credentials/ee-credentials.json"))

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [1]:
# Filter collection for the target event
fire_fc = ee.FeatureCollection("JRC/GWIS/GlobFire/v2/FinalPerimeters")
# fire_fc = fire_fc.filter(ee.Filter.eq("Id", "FL2754808157719850109"))

ss = fire_fc.first().getInfo()
print(ss)

# Option A: Format directly on the GEE server side
#end_date_str = ee.Date(fire_fc.first().get("FinalDate")).format("YYYY-MM-dd").getInfo()

#print("Fire End Date:", end_date_str)

NameError: name 'ee' is not defined

### Checking the revnet modified model


In [4]:
cfg = Config(Path("1_basin.yml"), dev_mode=True)
    # Add additional configuration
dict_config = cfg.as_dict()
dict_config.update({
        "daily_input_size":7, 
        "daily_hidden_size":20, 
        "burn_input_size":5,
        "burn_hidden_size1":4, 
        "burn_hidden_size2":3,
        "burn_kernel_size1":5, 
        "burn_kernel_size2":5, 
        # "burn_area_resolution_days":30, 
        "train_basin_file": "1_basin.txt",
        "data_dir":"../../data/CAMELS_US",
        "hidden_size":20, 
        "train_burn_start_dates":{"01013500":"1986-12"},
        "train_burn_end_dates":{"01013500":"2010-01"},
        "month_emb_dim":4,
        "model":"revnet",
        "dataset":"camels_us_burn"
        
        })
cfg = Config(dict_config, dev_mode=True) # Update config
    


### Load the dataset with burn data

In [5]:
dataset = CamelsUSBurn(cfg, is_train=True, period="train" )

INFO:Camels US:Loading burn timeseries data
INFO:neuralhydrology.datasetzoo.basedataset:Loading basin data into xarray data set.
100%|██████████| 119/119 [00:38<00:00,  3.07it/s]
INFO:Camels US:Finished loading flow and forcings time series data. Preparing to load burn timeseries data
INFO:Camels US:Finished loading burn time series data
INFO:Camels US:Creating lookup tables for forcings and flow data
INFO:neuralhydrology.datasetzoo.basedataset:Create lookup table and convert to pytorch tensor
100%|██████████| 119/119 [00:11<00:00, 10.56it/s]
INFO:Camels US:Finished creating lookup tables for forcings and flow data
INFO:Camels US:Enriching lookup tables with burn data
100%|██████████| 119/119 [00:01<00:00, 74.31it/s]
INFO:Camels US:Adding indexes for burn data corresponding to the end days in forcing date for each sample
100%|██████████| 391272/391272 [00:01<00:00, 206590.93it/s]
INFO:Camels US:Finished enriching lookup tables with burn data


In [6]:
dataloaded = [dataset[i] for i in np.random.choice(len(dataset), size=20)]
collated = dataset.collate_fn(dataloaded)    

In [7]:
# Train the model with existing trainer
start_training(cfg)


INFO:neuralhydrology.utils.logging_utils:Logging to /bsuhome/manishadhikari/goodfire/practice-neural-hydrology/neuralhydrology/notebooks/burndata/runs/test_run_2209_182347/output.log initialized.
INFO:neuralhydrology.training.basetrainer:### Folder structure created at /bsuhome/manishadhikari/goodfire/practice-neural-hydrology/neuralhydrology/notebooks/burndata/runs/test_run_2209_182347
INFO:neuralhydrology.training.basetrainer:### Run configurations for test_run
INFO:neuralhydrology.training.basetrainer:experiment_name: test_run
INFO:neuralhydrology.training.basetrainer:train_basin_file: 1_basin.txt
INFO:neuralhydrology.training.basetrainer:validation_basin_file: 1_basin.txt
INFO:neuralhydrology.training.basetrainer:test_basin_file: 1_basin.txt
INFO:neuralhydrology.training.basetrainer:train_start_date: 1999-10-01 00:00:00
INFO:neuralhydrology.training.basetrainer:train_end_date: 2008-09-30 00:00:00
INFO:neuralhydrology.training.basetrainer:validation_start_date: 1980-10-01 00:00:00
I

AttributeError: 'list' object has no attribute 'to'

# Make one run of the data

In [ ]:
revnet = ModifiedRevnet(cfg)
output = revnet(collated)
output

### Pytorch generated structure of revnet

In [ ]:
from torchview import draw_graph
model_graph = draw_graph(revnet, input_data=collated, device='meta')
model_graph.visual_graph

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/logs

### Excluded basin notes

03281100: NaN runoff ratio

I have not included most of the basins because it says Year is somehow missing.
I gotta diagnose it later